# Amiri pipeline

Real-life: **ssd** trim. Synthetic: **none** trim.

## Real

In [ ]:
import sys
import json
import math
import time
from pathlib import Path

import pandas as pd
import numpy as np
import pm4py
from pm4py.objects.log.exporter.xes import exporter as xes_exporter
from sklearn.metrics import mean_absolute_error, mean_squared_error

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "steady_state_detection"))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "plain-field"))

from ssd_trim import run_ssd_trim
from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
from create_prefixes_from_windows import make_three_way_split
from time_series_preprocessing import ts_splits_from_log
from run_predictions_real import _make_half_prefix_test_df
from amiri.trainer import AmiriTrainer
from amiri.params import default_params as amiri_default_params

REAL_DATA_DIR = ROOT / "data" / "real-life"
BEST_MODELS = ROOT / "best_models"
RESULTS = ROOT / "results"

REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
                "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]
TRIM_NAME = "ssd"


def load_data_for_ssd(xes_path):
    """Returns (df, cc, tt, train_df, val_df, test_df) for the ssd trim."""
    from time_series_preprocessing import Split3WayConfig, split_timeseries
    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from create_prefixes_from_windows import make_three_way_split
    from ssd_trim import run_ssd_trim

    log = pm4py.read_xes(str(xes_path))

    full_cc_raw = create_concurrent_cases_timeseries(log, plot=False)
    ssd_result = run_ssd_trim(log, window_step="D")
    canonical_end = ssd_result["cutoff"] if ssd_result["cutoff"] is not None else full_cc_raw.index[-1]
    full_cc_trimmed = full_cc_raw[full_cc_raw.index <= canonical_end]
    split_cfg = Split3WayConfig(train_frac=0.70, val_frac=0.10, test_frac=0.20)
    _, _, _, train_split, val_split = split_timeseries(full_cc_trimmed, split_cfg)

    full_tt_raw = create_avg_throughtput_time_timeseries(log, plot=False)
    full_tt_trimmed = full_tt_raw[full_tt_raw.index <= canonical_end]

    def _slice(raw, trimmed):
        idx = trimmed.index
        lo = train_split.tz_convert(None) if idx.tz is None else train_split
        hi = val_split.tz_convert(None) if idx.tz is None else val_split
        return {
            "raw": raw, "trimmed": trimmed,
            "train": trimmed[idx <= lo],
            "val": trimmed[(idx > lo) & (idx <= hi)],
            "test": trimmed[idx > hi],
            "train_split": train_split, "val_split": val_split,
        }

    cc = _slice(full_cc_raw, full_cc_trimmed)
    tt = _slice(full_tt_raw, full_tt_trimmed)

    df = pm4py.convert_to_dataframe(log)
    df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True)
    df = df.dropna(subset=["case:concept:name"])
    _cols = {"case:concept:name": "caseid", "concept:name": "task",
             "lifecycle:transition": "event_type", "time:timestamp": "end_timestamp"}
    _cols["org:resource" if "org:resource" in df.columns else "org:group"] = "user"
    df = df.rename(columns=_cols)
    df["task"] = df["task"].fillna("unk")
    df["user"] = df["user"].fillna("unk")

    train_, val_, test_ = make_three_way_split(
        df, case_col="caseid", time_col="end_timestamp",
        train_split=cc["train_split"], val_split=cc["val_split"], full_traces=True,
    )
    return df, cc, tt, train_, val_, test_
def rem_time_to_event_log(rt_df):
    rt = rt_df.copy()
    rt["start_timestamp"] = pd.to_datetime(rt["start_timestamp"])
    rt["anchor_timestamp"] = pd.to_datetime(rt["anchor_timestamp"])
    rt["predicted_end"] = rt["anchor_timestamp"] + pd.to_timedelta(rt["rem_time_days"], unit="D")
    return pd.concat([
        rt[["caseid", "start_timestamp"]].rename(columns={"start_timestamp": "end_timestamp"}),
        rt[["caseid", "predicted_end"]].rename(columns={"predicted_end": "end_timestamp"}),
    ], ignore_index=True)


def save_amiri_metrics(out_dir: Path, run_name: str, cc_test, cc_pred, tt_test, tt_pred):
    out_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([
        dict(dataset=run_name, series="concurrent_cases", model="amiri",
             mae=mean_absolute_error(cc_test, cc_pred), mse=mean_squared_error(cc_test, cc_pred)),
        dict(dataset=run_name, series="throughput_time", model="amiri",
             mae=mean_absolute_error(tt_test, tt_pred), mse=mean_squared_error(tt_test, tt_pred)),
    ]).to_csv(out_dir / f"metrics_{run_name}.csv", index=False)


print(f"{len(REAL_DATASETS)} real-life datasets")

### 1. First (full-trace)

In [ ]:
from amiri.converter import convert
from amiri.trainer import select_amiri_hpo_winner
from hpo_val_scoring import build_val_as_test_df
from hyperopt import tpe, Trials, hp, fmin, STATUS_OK
import pickle, shutil

AMIRI_MAX_EVALS = 12

AMIRI_SPACE = {
    "gt_layers":       hp.choice("gt_layers",    [3, 5, 7]),
    "gt_n_heads":      hp.choice("gt_n_heads",   [4, 8]),
    "gt_dim_hidden":   hp.choice("gt_dim_hidden", [32, 64, 128]),
    "gt_dropout":      hp.uniform("gt_dropout",  0.0, 0.4),
    "gt_attn_dropout": hp.uniform("gt_attn_dropout", 0.0, 0.6),
    "base_lr":         hp.loguniform("base_lr",  math.log(1e-4), math.log(1e-2)),
    "weight_decay":    hp.loguniform("weight_decay", math.log(1e-4), math.log(1e-1)),
    "batch_size":      hp.choice("batch_size",   [64, 128, 256]),
    "max_epoch":       hp.choice("max_epoch",    [50, 100]),
}

amiri_data = {}

for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (amiri full-trace HPO)\n{'='*60}")
    xes_path = REAL_DATA_DIR / f"{name}.xes"
    df_full, cc, tt, train_df, val_df, test_df = load_data_for_ssd(xes_path)

    run_name = f"{name}_test_full"
    RUN_NAME = f"{name}_full"
    hpo_dir = BEST_MODELS / name / TRIM_NAME / "amiri" / "hpo_trials"
    hpo_dir.mkdir(parents=True, exist_ok=True)
    shared_dataset_dir = hpo_dir / "shared_dataset"
    shared_dataset_dir.mkdir(parents=True, exist_ok=True)

    best_model_dir = BEST_MODELS / name / TRIM_NAME / "amiri" / RUN_NAME
    amiri_data[name] = (df_full, cc, tt, train_df, val_df, test_df, best_model_dir, shared_dataset_dir)

    metrics_path = RESULTS / "amiri_hpo" / TRIM_NAME / run_name / f"metrics_{run_name}.csv"
    if metrics_path.exists():
        print(f"  [skip] full-trace metrics already exist -> {metrics_path}")
        continue

    _raw_meta = shared_dataset_dir / "AMIRI" / "raw" / "meta.pkl"
    if not _raw_meta.exists():
        print("  Converting data to PyG graphs (one-time)...")
        convert(train_df, val_df, test_df, shared_dataset_dir, seed=42)
    else:
        print(f"  Graph data already exists at {shared_dataset_dir}")

    _trials_pkl = hpo_dir / "hyperopt_trials.pkl"
    if _trials_pkl.exists():
        with open(_trials_pkl, "rb") as f:
            _hpo_trials = pickle.load(f)
        _completed = len(_hpo_trials.trials)
        print(f"  Resuming HPO: {_completed} trials already done.")
    else:
        _hpo_trials = Trials()
        _completed = 0
        print("  Starting fresh HPO.")

    amiri_hpo_results = []
    _trial_counter = [_completed]

    def _amiri_objective(trial_cfg, _hpo_dir=hpo_dir, _trials_pkl=_trials_pkl,
                         _hpo_trials=_hpo_trials, _trial_counter=_trial_counter,
                         _shared_dataset_dir=shared_dataset_dir, _train_df=train_df,
                         _val_df=val_df, _test_df=test_df, _results=amiri_hpo_results):
        i = _trial_counter[0]
        _trial_counter[0] += 1
        trial_id = f"trial_{i:03d}"
        trial_dir = _hpo_dir / trial_id
        trial_dir.mkdir(parents=True, exist_ok=True)

        with open(_trials_pkl, "wb") as f:
            pickle.dump(_hpo_trials, f)

        cfg_path = trial_dir / "config.json"
        if not cfg_path.exists():
            cfg_path.write_text(json.dumps(trial_cfg, indent=2))

        result_path = trial_dir / "result.json"
        if result_path.exists():
            res = json.loads(result_path.read_text())
            print(f"    [{trial_id}] cached val_mae={res['val_mae']:.4f}")
            _results.append(res)
            return {"loss": res["val_mae"], "status": STATUS_OK, **res}

        params = amiri_default_params(**trial_cfg)
        params["seed"] = 42

        trainer = AmiriTrainer(
            _train_df, _val_df, _test_df,
            run_name=trial_id, params=params,
            output_dir=trial_dir, dataset_dir=_shared_dataset_dir,
        )
        try:
            trainer.run()
            val_mae = trainer.get_best_val_mae()
        except Exception as e:
            print(f"    [{trial_id}] FAILED: {e}")
            res = {"trial_id": trial_id, "val_mae": float("inf"), "error": str(e), **trial_cfg}
            result_path.write_text(json.dumps(res, indent=2))
            _results.append(res)
            return {"loss": float("inf"), "status": STATUS_OK, **res}

        res = {"trial_id": trial_id, "val_mae": val_mae, **trial_cfg}
        result_path.write_text(json.dumps(res, indent=2))
        _results.append(res)
        print(f"    [{trial_id}] val_mae={val_mae:.4f}")
        return {"loss": val_mae, "status": STATUS_OK, **res}


    best_amiri = fmin(
        fn=_amiri_objective, space=AMIRI_SPACE, algo=tpe.suggest,
        max_evals=AMIRI_MAX_EVALS, trials=_hpo_trials, verbose=True,
    )
    with open(_trials_pkl, "wb") as f:
        pickle.dump(_hpo_trials, f)
    print(f"  HPO complete for {name}.")


    val_as_test_df = build_val_as_test_df(df_full, cc["train_split"])
    winner = select_amiri_hpo_winner(
        hpo_dir, train_df, val_df, val_as_test_df, cc["val"], tt["val"],
        default_params_fn=amiri_default_params,
    )
    if winner is None:
        raise RuntimeError(f"val-CC-MAE rescoring found no usable trial for {name}.")

    best_model_dir.mkdir(parents=True, exist_ok=True)
    _gps_src = winner["gps_results_dir"]
    _gps_dst = best_model_dir / "gps_results"
    if _gps_src.exists() and not _gps_dst.exists():
        shutil.copytree(_gps_src, _gps_dst)
        print(f"  Copied winning trial's GPS results -> {_gps_dst}")

    _best_params = amiri_default_params(**{
        k: v.item() if hasattr(v, "item") else v
        for k, v in winner["cfg"].items()
    })
    _best_params["seed"] = 42
    _best_params["selection_metric"] = "val_cc_mae"
    _best_params["val_cc_mae"] = winner["val_cc_mae"]
    _best_params["val_tt_mae"] = winner["val_tt_mae"]
    (best_model_dir / "best_params.json").write_text(json.dumps(_best_params, indent=2))
    print(f"  Winning trial: {winner['trial_id']}  val_cc_mae={winner['val_cc_mae']:.4f}  val_tt_mae={winner['val_tt_mae']:.4f}")

    final_trainer = AmiriTrainer(
        train_df, val_df, test_df, run_name=RUN_NAME,
        params=_best_params, output_dir=best_model_dir, dataset_dir=shared_dataset_dir,
    )
    _ckpt_base = best_model_dir / "gps_results" / "amiri_gps"
    if not _ckpt_base.exists():
        print("  Training best model (weights not copied from trial, retraining)...")
        final_trainer.run()
    else:
        print("  Using existing GPS results copied from best trial.")

    rem_time_df = final_trainer.predict()
    event_log_a = rem_time_to_event_log(rem_time_df)

    cc_pred = (create_concurrent_cases_timeseries(event_log_a, time_col="end_timestamp", case_col="caseid",
                                                  window="days", plot=False)
              .reindex(cc["test"].index).ffill().bfill().fillna(0))
    tt_pred = (create_avg_throughtput_time_timeseries(event_log_a, time_col="end_timestamp", case_col="caseid",
                                                       window="days", plot=False)
              .reindex(tt["test"].index).ffill().bfill().fillna(0))

    save_amiri_metrics(metrics_path.parent, run_name, cc["test"], cc_pred, tt["test"], tt_pred)
    print(f"  saved -> {metrics_path.parent}")

### 2. Half-prefix

In [ ]:
for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (amiri half-prefix)\n{'='*60}")
    if name not in amiri_data:
        print("  [skip] run section 1 first")
        continue
    df_full, cc, tt, train_df, val_df, test_df, amiri_dir, dataset_dir = amiri_data[name]

    if not (amiri_dir / "gps_results").exists():
        print(f"  [skip] no trained model at {amiri_dir}")
        continue

    run_name = f"{name}_test_full"
    metrics_path = RESULTS / "amiri_hpo_half" / TRIM_NAME / run_name / f"metrics_{run_name}.csv"
    if metrics_path.exists():
        print(f"  [skip] half-prefix metrics already exist -> {metrics_path}")
        continue

    test_df_half = _make_half_prefix_test_df(df_full, test_df)

    half_dir = amiri_dir / "half_prefix"
    half_dataset_dir = half_dir / "dataset"
    half_dir.mkdir(parents=True, exist_ok=True)
    gps_link = half_dir / "gps_results"
    if not gps_link.exists():
        gps_link.symlink_to(amiri_dir / "gps_results")


    best_params = json.loads((amiri_dir / "best_params.json").read_text())
    trainer_half = AmiriTrainer(train_df, val_df, test_df_half, run_name=name,
                                params=best_params, output_dir=half_dir,
                                dataset_dir=half_dataset_dir, ref_dataset_dir=dataset_dir)
    rem_time_df = trainer_half.predict()
    event_log_a = rem_time_to_event_log(rem_time_df)

    cc_pred = (create_concurrent_cases_timeseries(event_log_a, time_col="end_timestamp", case_col="caseid",
                                                  window="days", plot=False)
              .reindex(cc["test"].index).ffill().bfill().fillna(0))
    tt_pred = (create_avg_throughtput_time_timeseries(event_log_a, time_col="end_timestamp", case_col="caseid",
                                                       window="days", plot=False)
              .reindex(tt["test"].index).ffill().bfill().fillna(0))

    save_amiri_metrics(metrics_path.parent, run_name, cc["test"], cc_pred, tt["test"], tt_pred)
    print(f"  saved -> {metrics_path.parent}")

### 3. Plain-field

In [ ]:
from runner import (
    get_inflight_cases, build_sos_cases, predict_amiri_plain_field,
    compute_cc_tt_metrics, save_pf_metrics, save_pf_plot, save_arrival_eval,
)
from sos import most_frequent_first_activity, most_frequent_first_resource, empirical_arrival_hour_sampler
from arrival import compute_arrival_series, ProphetArrivalModel

for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (amiri plain-field)\n{'='*60}")
    if name not in amiri_data:
        print("  [skip] run section 1 first")
        continue
    df_full, cc, tt, train_df, val_df, test_df, amiri_dir, dataset_dir = amiri_data[name]

    if not (amiri_dir / "gps_results").exists():
        print(f"  [skip] no trained model at {amiri_dir}")
        continue

    out = RESULTS / "plain_field" / "amiri" / TRIM_NAME / f"{name}_test_full"
    if (out / f"metrics_{name}_test_full.csv").exists():
        print(f"  [skip] plain-field metrics already exist -> {out}")
        continue

    val_split = cc["val_split"]
    inflight_df = get_inflight_cases(df_full, val_split, case_col="caseid", time_col="end_timestamp")


    known_df = pd.concat([train_df, val_df], ignore_index=True)
    val_split_ts = pd.Timestamp(val_split)
    if val_split_ts.tzinfo:
        val_split_ts = val_split_ts.tz_convert(None)
    arrivals_full = compute_arrival_series(df_full, case_col="caseid", time_col="end_timestamp")
    arrivals = arrivals_full[arrivals_full.index < val_split_ts]
    arrival_model = ProphetArrivalModel().fit(arrivals)
    predicted_arrivals = arrival_model.predict(pd.DatetimeIndex(cc["test"].index).tz_localize(None))
    save_arrival_eval(predicted_arrivals, arrivals_full, cc["test"].index, out, name, TRIM_NAME)
    sos_df = build_sos_cases(
        predicted_arrivals,
        most_frequent_first_activity(known_df), most_frequent_first_resource(known_df),
        empirical_arrival_hour_sampler(known_df),
    )


    best_params = json.loads((amiri_dir / "best_params.json").read_text())
    trainer_a = AmiriTrainer(train_df, val_df, test_df, run_name=f"{name}_test_full",
                             params=best_params, output_dir=amiri_dir, dataset_dir=dataset_dir)
    rem_time_df_a = predict_amiri_plain_field(trainer_a, sos_df, inflight_df)
    event_log_a = rem_time_to_event_log(rem_time_df_a)

    cc_p, tt_p, m = compute_cc_tt_metrics(event_log_a, cc["test"], tt["test"])
    save_pf_metrics(name, "plain_field_amiri", m, out)
    save_pf_plot(cc_p, tt_p, cc["test"], tt["test"], out, name, TRIM_NAME, "plain_field_amiri")
    print(f"  [amiri pf] cc_mae={m['cc_mae']:.2f}  tt_mae={m['tt_mae']:.2f}")

## Synthetic

In [ ]:
import warnings, math, pickle
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from hyperopt import tpe, Trials, hp, fmin, STATUS_OK
from time_series_preprocessing import ts_splits_from_log, Split3WayConfig
from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
from create_prefixes_from_windows import load_event_log, make_three_way_split
from setttings import set_global_seed
from hpo_val_scoring import build_val_as_test_df
from amiri.trainer import AmiriTrainer, select_amiri_hpo_winner
from amiri.params  import default_params
from amiri.converter import convert

_COLS = {
    'case:concept:name':    'caseid',
    'concept:name':         'task',
    'lifecycle:transition': 'event_type',
    'org:resource':         'user',
    'time:timestamp':       'end_timestamp',
}
_TRIM = 'none'
AMIRI_MAX_EVALS = 12

def rem_time_to_event_log(rem_df):
    """Build a 2-event-per-case log: case start + predicted end."""
    rt = rem_df.copy()
    rt['start_timestamp']  = pd.to_datetime(rt['start_timestamp'])
    rt['anchor_timestamp'] = pd.to_datetime(rt['anchor_timestamp'])
    rt['predicted_end']    = rt['anchor_timestamp'] + pd.to_timedelta(rt['rem_time_days'], unit='D')
    return pd.concat([
        rt[['caseid', 'start_timestamp']].rename(columns={'start_timestamp': 'end_timestamp'}),
        rt[['caseid', 'predicted_end']].rename(columns={'predicted_end': 'end_timestamp'}),
    ], ignore_index=True)

def pt_kpi_series(event_log, test_index_cc, test_index_tt, window='days'):
    pred_cc = create_concurrent_cases_timeseries(
        event_log, time_col='end_timestamp', case_col='caseid', window=window, plot=False)
    pred_tt = create_avg_throughtput_time_timeseries(
        event_log, time_col='end_timestamp', case_col='caseid', window=window, plot=False)
    cc_arr = pred_cc.reindex(test_index_cc).ffill().bfill().fillna(0).to_numpy()
    tt_arr = pred_tt.reindex(test_index_tt).ffill().bfill().fillna(0).to_numpy()
    return cc_arr, tt_arr

set_global_seed(1904)

_EXCLUDE_DATASETS = {'loan_recency', 'o2c_recency'}
SYNTH_DATASETS = sorted(
    p for p in (ROOT / 'data' / 'synthetic').glob('*.xes')
    if p.stem not in _EXCLUDE_DATASETS
)
print(f'{len(SYNTH_DATASETS)} synthetic datasets: {[p.stem for p in SYNTH_DATASETS]}')


### 1. First (full-trace)

In [ ]:
AMIRI_SPACE = {
    'gt_layers':        hp.choice('gt_layers',    [3, 5, 7]),
    'gt_n_heads':       hp.choice('gt_n_heads',   [4, 8]),
    'gt_dim_hidden':    hp.choice('gt_dim_hidden', [32, 64, 128]),
    'gt_dropout':       hp.uniform('gt_dropout',  0.0, 0.4),
    'gt_attn_dropout':  hp.uniform('gt_attn_dropout', 0.0, 0.6),
    'base_lr':          hp.loguniform('base_lr',  np.log(1e-4), np.log(1e-2)),
    'weight_decay':     hp.loguniform('weight_decay', np.log(1e-4), np.log(1e-1)),
    'batch_size':       hp.choice('batch_size',   [64, 128, 256]),
    'max_epoch':        hp.choice('max_epoch',    [50, 100]),
}

for _xes_path in SYNTH_DATASETS:
    DATASET_NAME = _xes_path.stem
    RUN_NAME     = f'{DATASET_NAME}_full'
    print(f"\n{'='*60}\n{DATASET_NAME} (amiri synthetic HPO + first)\n{'='*60}")

    log = pm4py.read_xes(str(_xes_path))
    ts = ts_splits_from_log(
        log,
        trim_method=None, trim_pct=0.25, trim_k=1.5, trim_frac=0.5, trim_window=7,
        train_frac=0.7, val_frac=0.1,
        cut_date=None,
    )
    cc = ts['concurrent_cases']
    tt = ts['throughput_time']

    train_split = cc['train_split']
    val_split   = cc['val_split']

    df_raw = load_event_log(_xes_path, time_col='time:timestamp', case_col='case:concept:name')
    df = df_raw.rename(columns=_COLS)
    df['task'] = df['task'].fillna('unk')
    df['user'] = df['user'].fillna('unk')

    train_df, val_df, test_df = make_three_way_split(
        df, case_col='caseid', time_col='end_timestamp',
        train_split=train_split, val_split=val_split, full_traces=True,
    )

    val_as_test_df = build_val_as_test_df(df, train_split)

    hpo_dir = BEST_MODELS / DATASET_NAME / 'amiri' / _TRIM / 'hpo_trials'
    hpo_dir.mkdir(parents=True, exist_ok=True)
    shared_dataset_dir = hpo_dir / 'shared_dataset'
    shared_dataset_dir.mkdir(parents=True, exist_ok=True)

    _trials_pkl = hpo_dir / 'hyperopt_trials.pkl'
    if _trials_pkl.exists():
        with open(_trials_pkl, 'rb') as _f:
            _hpo_trials = pickle.load(_f)
        _completed = len(_hpo_trials.trials)
        print(f'Resuming HPO: {_completed} trials already done.')
    else:
        _hpo_trials = Trials()
        _completed  = 0
        print('Starting fresh HPO.')

    _raw_meta = shared_dataset_dir / 'AMIRI' / 'raw' / 'meta.pkl'
    if not _raw_meta.exists():
        print('Converting data to PyG graphs (one-time) …')
        convert(train_df, val_df, test_df, shared_dataset_dir, seed=42)
    else:
        print(f'Graph data already exists at {shared_dataset_dir}')

    amiri_hpo_results = []
    _trial_counter    = [_completed]

    def _amiri_objective(trial_cfg):
        i = _trial_counter[0]
        _trial_counter[0] += 1
        trial_id  = f'trial_{i:03d}'
        trial_dir = hpo_dir / trial_id
        trial_dir.mkdir(parents=True, exist_ok=True)

        with open(_trials_pkl, 'wb') as _f:
            pickle.dump(_hpo_trials, _f)

        cfg_path = trial_dir / 'config.json'
        if not cfg_path.exists():
            cfg_path.write_text(json.dumps(trial_cfg, indent=2))

        result_path = trial_dir / 'result.json'
        if result_path.exists():
            res = json.loads(result_path.read_text())
            print(f'[{trial_id}] cached val_mae={res["val_mae"]:.4f}')
            amiri_hpo_results.append(res)
            return {'loss': res['val_mae'], 'status': STATUS_OK, **res}

        params = default_params(**trial_cfg)
        params['seed'] = 42

        trainer = AmiriTrainer(
            train_df, val_df, test_df,
            run_name=trial_id,
            params=params,
            output_dir=trial_dir,
            dataset_dir=shared_dataset_dir,
        )
        try:
            trainer.run()
            val_mae = trainer.get_best_val_mae()
        except Exception as e:
            print(f'[{trial_id}] FAILED: {e}')
            res = {'trial_id': trial_id, 'val_mae': float('inf'), 'error': str(e), **trial_cfg}
            result_path.write_text(json.dumps(res, indent=2))
            amiri_hpo_results.append(res)
            return {'loss': float('inf'), 'status': STATUS_OK, **res}

        res = {'trial_id': trial_id, 'val_mae': val_mae, **trial_cfg}
        result_path.write_text(json.dumps(res, indent=2))
        amiri_hpo_results.append(res)
        print(f'[{trial_id}] val_mae={val_mae:.4f}')
        return {'loss': val_mae, 'status': STATUS_OK, **res}

    best_amiri = fmin(
        fn=_amiri_objective, space=AMIRI_SPACE, algo=tpe.suggest,
        max_evals=AMIRI_MAX_EVALS, trials=_hpo_trials, verbose=True,
    )
    with open(_trials_pkl, 'wb') as _f:
        pickle.dump(_hpo_trials, _f)
    print(f'HPO complete for {DATASET_NAME}.')

    if not amiri_hpo_results:
        for _rp in sorted(hpo_dir.glob('trial_*/result.json')):
            try:
                amiri_hpo_results.append(json.loads(_rp.read_text()))
            except Exception:
                pass
        print(f'Loaded {len(amiri_hpo_results)} trial results from disk.')


    winner = select_amiri_hpo_winner(
        hpo_dir, train_df, val_df, val_as_test_df, cc['val'], tt['val'],
        default_params_fn=default_params,
    )
    if winner is None:
        raise RuntimeError(f'val-CC-MAE rescoring found no usable trial for {DATASET_NAME}.')

    best_model_dir = BEST_MODELS / DATASET_NAME / 'amiri' / _TRIM / RUN_NAME
    best_model_dir.mkdir(parents=True, exist_ok=True)

    _gps_src = winner['gps_results_dir']
    _gps_dst = best_model_dir / 'gps_results'
    if _gps_src.exists() and not _gps_dst.exists():
        shutil.copytree(_gps_src, _gps_dst)
        print(f'Copied winning GPS results -> {_gps_dst}')

    _best_params = default_params(**{
        k: v.item() if hasattr(v, 'item') else v
        for k, v in winner['cfg'].items()
    })
    _best_params['seed'] = 42
    _best_params['selection_metric'] = 'val_cc_mae'
    _best_params['val_cc_mae'] = winner['val_cc_mae']
    _best_params['val_tt_mae'] = winner['val_tt_mae']

    _params_path = best_model_dir / 'best_params.json'
    _params_path.write_text(json.dumps(_best_params, indent=2))
    print(f'Winning trial: {winner["trial_id"]}  val_cc_mae={winner["val_cc_mae"]:.4f}  val_tt_mae={winner["val_tt_mae"]:.4f}')

    final_trainer = AmiriTrainer(
        train_df, val_df, test_df,
        run_name=RUN_NAME,
        params=_best_params,
        output_dir=best_model_dir,
        dataset_dir=shared_dataset_dir,
    )
    _ckpt_base = best_model_dir / 'gps_results' / 'amiri_gps'
    if not _ckpt_base.exists():
        print('Training best model …')
        final_trainer.run()
    else:
        print('Using existing GPS results from best trial.')

    rem_time_df = final_trainer.predict()
    event_log_a = rem_time_to_event_log(rem_time_df)
    cc_pred = (create_concurrent_cases_timeseries(event_log_a, time_col='end_timestamp', case_col='caseid',
                                                  window='days', plot=False)
              .reindex(cc['test'].index).ffill().bfill().fillna(0))
    tt_pred = (create_avg_throughtput_time_timeseries(event_log_a, time_col='end_timestamp', case_col='caseid',
                                                       window='days', plot=False)
              .reindex(tt['test'].index).ffill().bfill().fillna(0))

    metrics_path = RESULTS / 'amiri_hpo' / _TRIM / RUN_NAME / f'metrics_{RUN_NAME}.csv'
    save_amiri_metrics(metrics_path.parent, RUN_NAME, cc['test'], cc_pred, tt['test'], tt_pred)
    print(f'Saved -> {metrics_path.parent}')


### 2. Half-prefix

In [ ]:
import shutil

for _xes_path in SYNTH_DATASETS:
    DATASET_NAME = _xes_path.stem
    RUN_NAME     = f'{DATASET_NAME}_full'

    best_model_dir = BEST_MODELS / DATASET_NAME / 'amiri' / _TRIM / RUN_NAME
    best_params_path = best_model_dir / 'best_params.json'
    if not best_params_path.exists():
        print(f'[skip] {DATASET_NAME}: no trained model')
        continue

    metrics_path = RESULTS / 'amiri_hpo_half' / _TRIM / RUN_NAME / f'metrics_{RUN_NAME}.csv'
    if metrics_path.exists():
        print(f'[skip] {DATASET_NAME}: half-prefix results exist')
        continue

    print(f"\n{'='*60}\n{DATASET_NAME} (amiri synthetic half-prefix)\n{'='*60}")

    log = pm4py.read_xes(str(_xes_path))
    ts = ts_splits_from_log(
        log,
        trim_method=None, trim_pct=0.25, trim_k=1.5, trim_frac=0.6, trim_window=7,
        train_frac=0.7, val_frac=0.1,
        cut_date=None,
    )
    cc = ts['concurrent_cases']
    tt = ts['throughput_time']

    df_raw = load_event_log(_xes_path, time_col='time:timestamp', case_col='case:concept:name')
    df = df_raw.rename(columns=_COLS)
    df['task'] = df['task'].fillna('unk')
    df['user'] = df['user'].fillna('unk')

    train_split = cc['train_split']
    val_split   = cc['val_split']
    train_df, val_df, test_df_std = make_three_way_split(
        df, case_col='caseid', time_col='end_timestamp',
        train_split=train_split, val_split=val_split, full_traces=True,
    )

    _test_cids    = set(test_df_std['caseid'].astype(str))
    _df_test_full = (df[df['caseid'].astype(str).isin(_test_cids)]
                     .copy().sort_values(['caseid', 'end_timestamp']))
    _half_parts = []
    for _cid, _grp in _df_test_full.groupby('caseid', sort=False):
        _n = len(_grp)
        _half_parts.append(_grp.iloc[: max(1, math.ceil(_n / 2))])
    test_df = pd.concat(_half_parts, ignore_index=True)

    cc_actual = cc['test'].to_numpy()
    tt_actual = tt['test'].to_numpy()

    half_dir         = best_model_dir / 'half_prefix'
    half_dataset_dir = half_dir / 'dataset'
    half_dir.mkdir(parents=True, exist_ok=True)

    _gps_link = half_dir / 'gps_results'
    if not _gps_link.exists():
        _gps_link.symlink_to(best_model_dir / 'gps_results')

    _rem_time_path = half_dir / 'rem_time.csv'
    if _rem_time_path.exists():
        print('  loading cached rem_time.csv')
        rem_time_df = pd.read_csv(_rem_time_path,
                                   parse_dates=['start_timestamp', 'anchor_timestamp'])
        predict_s_a = 0.0
    else:
        best_params = json.loads(best_params_path.read_text())
        trainer_a = AmiriTrainer(
            train_df, val_df, test_df,
            run_name=DATASET_NAME,
            params=best_params,
            output_dir=half_dir,
            dataset_dir=half_dataset_dir,
        )
        t0 = time.perf_counter()
        rem_time_df = trainer_a.predict()
        predict_s_a = round(time.perf_counter() - t0, 2)

    event_log_a = rem_time_to_event_log(rem_time_df)
    cc_pred_a, tt_pred_a = pt_kpi_series(
        event_log_a, test_index_cc=cc['test'].index, test_index_tt=tt['test'].index)

    out_dir_a = metrics_path.parent
    out_dir_a.mkdir(parents=True, exist_ok=True)

    results_a = {
        'concurrent_cases': {'mse': mean_squared_error(cc_actual, cc_pred_a),
                             'mae': mean_absolute_error(cc_actual, cc_pred_a)},
        'throughput_time':  {'mse': mean_squared_error(tt_actual, tt_pred_a),
                             'mae': mean_absolute_error(tt_actual, tt_pred_a)},
    }
    pd.DataFrame([
        dict(dataset=RUN_NAME, series=s, model='amiri_hpo_half', mse=m['mse'], mae=m['mae'])
        for s, m in results_a.items()
    ]).to_csv(out_dir_a / f'metrics_{RUN_NAME}.csv', index=False)

    for series_name, actual, pred, index in [
        ('concurrent_cases', cc_actual, cc_pred_a, cc['test'].index),
        ('throughput_time',  tt_actual, tt_pred_a, tt['test'].index),
    ]:
        m = results_a[series_name]
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.plot(index, actual, color='green',  label='actual',                linewidth=1.5)
        ax.plot(index, pred,   color='purple', label='Amiri-HPO-half (pred)', linestyle='--')
        ax.set_title(f'{RUN_NAME} — {series_name} (half-prefix)  MSE={m["mse"]:.4f}  MAE={m["mae"]:.4f}')
        ax.legend(); plt.tight_layout()
        plt.savefig(out_dir_a / f'{series_name}.png', dpi=150, bbox_inches='tight')
        plt.show(); plt.close()
    print(f'  {predict_s_a:.1f}s  ->  {out_dir_a}')


### 3. Plain-field

In [ ]:
from pathlib import Path
import sys, warnings
warnings.filterwarnings('ignore')

ROOT = Path('.').resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "plain-field"))

from pf_lukas_prediction import run_pf_job, SYNTH_LOGS

for ds in SYNTH_LOGS:
    run_pf_job(ds, "none", is_real=False, do_bukhsh=False, do_camargo=False)
